# Phase 1 - Step 0: Data Collection & Extraction


In [1]:
import os

for dataset in os.listdir('/kaggle/input/datasets'):
    print(f"\n{'='*50}")
    print(f"📁 DATASET: {dataset}")
    print('='*50)
    for root, dirs, files in os.walk(f'/kaggle/input/datasets/{dataset}'):
        level = root.replace(f'/kaggle/input/datasets/{dataset}', '').count(os.sep)
        if level > 4:
            continue
        indent = '  ' * level
        print(f'{indent}{os.path.basename(root)}/')
        for f in files[:2]:
            print(f'{indent}  📄 {f}')


📁 DATASET: aaryapatel98
aaryapatel98/
  indian-diabetic-retinopathy-image-dataset/
    A.%20Segmentation/
      A. Segmentation/
        📄 LICENSE.txt
        📄 CC-BY-4.0.txt
        2. All Segmentation Groundtruths/
        1. Original Images/
    B.%20Disease%20Grading/
      B. Disease Grading/
        📄 LICENSE.txt
        📄 CC-BY-4.0.txt
        2. Groundtruths/
          📄 a. IDRiD_Disease Grading_Training Labels.csv
          📄 b. IDRiD_Disease Grading_Testing Labels.csv
        1. Original Images/
    C.%20Localization/
      C. Localization/
        📄 LICENSE.txt
        📄 CC-BY-4.0.txt
        2. Groundtruths/
        1. Original Images/

📁 DATASET: pradosh123
pradosh123/
  retinal-vessel-segmentation-combined/
    Retina Vessel Segmentation/
      Training/
        Images/
          📄 Image_13R_CHASE.jpg
          📄 21_DRIVE.tif
        Masks/
          📄 06_h_HRF.tif
          📄 22_DRIVE.gif
      Test/
        Images/
          📄 Image_02L_CHASE.jpg
          📄 14_dr_HRF.

In [2]:
import os
import shutil
import numpy as np
from pathlib import Path
from PIL import Image

# ── Output folders ──────────────────────────────────────────────────────────
out_images       = Path('/kaggle/working/dataset_stage1/images')
out_vessel_masks = Path('/kaggle/working/dataset_stage1/vessel_masks')
out_lesion_masks = Path('/kaggle/working/dataset_stage1/lesion_masks')

for folder in [out_images, out_vessel_masks, out_lesion_masks]:
    folder.mkdir(parents=True, exist_ok=True)

print("✅ Output folders created:")
print(f"   {out_images}")
print(f"   {out_vessel_masks}")
print(f"   {out_lesion_masks}")

✅ Output folders created:
   /kaggle/working/dataset_stage1/images
   /kaggle/working/dataset_stage1/vessel_masks
   /kaggle/working/dataset_stage1/lesion_masks


In [3]:
def copy_and_rename(src_folder, dst_folder, prefix,
                    extensions=('.png', '.jpg', '.jpeg', '.tif', '.bmp')):
    """
    Copy all images from src_folder into dst_folder
    and rename them with a prefix so names never clash.
    Example: image_01.png → RETINOMIX_image_01.png
    """
    src = Path(src_folder)
    dst = Path(dst_folder)

    if not src.exists():
        print(f"❌ NOT FOUND: {src}")
        return 0

    copied = 0
    for file in sorted(src.iterdir()):
        if file.suffix.lower() in extensions:
            shutil.copy2(file, dst / f"{prefix}_{file.name}")
            copied += 1

    print(f"✅  {prefix:<35} → {copied} files copied")
    return copied

In [4]:
# RETINOMIX has flat images/ and mask/ folders — use those directly
BASE_RETINOMIX = '/kaggle/input/datasets/parikshakaur/retinomix-5/RETINOMIX-5'

copy_and_rename(
    src_folder = f'{BASE_RETINOMIX}/images',
    dst_folder = out_images,
    prefix     = 'RETINOMIX'
)
copy_and_rename(
    src_folder = f'{BASE_RETINOMIX}/mask',
    dst_folder = out_vessel_masks,
    prefix     = 'RETINOMIX'
)

✅  RETINOMIX                           → 233 files copied
✅  RETINOMIX                           → 233 files copied


233

In [5]:
# Capital I in Images and Masks — confirmed from data explorer
BASE_VESSEL = '/kaggle/input/datasets/pradosh123/retinal-vessel-segmentation-combined/Retina Vessel Segmentation'

# Training
copy_and_rename(
    src_folder = f'{BASE_VESSEL}/Training/Images',
    dst_folder = out_images,
    prefix     = 'VESSEL_train'
)
copy_and_rename(
    src_folder = f'{BASE_VESSEL}/Training/Masks',
    dst_folder = out_vessel_masks,
    prefix     = 'VESSEL_train'
)

# Test labeled only — skip Test Unlabeled
copy_and_rename(
    src_folder = f'{BASE_VESSEL}/Test/Images',
    dst_folder = out_images,
    prefix     = 'VESSEL_test'
)
copy_and_rename(
    src_folder = f'{BASE_VESSEL}/Test/Masks',
    dst_folder = out_vessel_masks,
    prefix     = 'VESSEL_test'
)

✅  VESSEL_train                        → 73 files copied
✅  VESSEL_train                        → 58 files copied
✅  VESSEL_test                         → 20 files copied
✅  VESSEL_test                         → 15 files copied


15

In [6]:
# VESSEL has 20 images from HRF subset that have no masks — remove them
vessel_imgs  = sorted([f for f in out_images.iterdir()
                        if f.name.startswith('VESSEL')])
vessel_masks = {f.stem: f for f in out_vessel_masks.iterdir()
                if f.name.startswith('VESSEL')}

removed = 0
for img in vessel_imgs:
    if img.stem not in vessel_masks:
        os.remove(img)
        removed += 1

print(f"✅ Removed {removed} VESSEL images with no mask")
print(f"   Remaining VESSEL images: {len([f for f in out_images.iterdir() if f.name.startswith('VESSEL')])}")

✅ Removed 20 VESSEL images with no mask
   Remaining VESSEL images: 73


In [7]:
BASE_IDRID = '/kaggle/input/datasets/aaryapatel98/indian-diabetic-retinopathy-image-dataset/A.%20Segmentation/A. Segmentation'

total_idrid_imgs = 0

for split_folder, tag in [('a. Training Set', 'train'),
                           ('b. Testing Set',  'test')]:

    img_dir = Path(BASE_IDRID) / '1. Original Images' / split_folder

    if not img_dir.exists():
        print(f"❌ Not found: {img_dir}")
        continue

    for img_file in sorted(img_dir.iterdir()):
        if img_file.suffix.lower() in ('.png', '.jpg', '.jpeg'):
            shutil.copy2(
                img_file,
                out_images / f'IDRID_{tag}_{img_file.name}'
            )
            total_idrid_imgs += 1

print(f"✅ IDRiD images copied: {total_idrid_imgs}")

✅ IDRiD images copied: 81


In [8]:
LESION_TYPES = {
    '1. Microaneurysms' : 'MA',
    '2. Haemorrhages'   : 'HE',
    '3. Hard Exudates'  : 'EX',
    '4. Soft Exudates'  : 'SE',
    '5. Optic Disc'     : 'OD'
}

# Create subfolders
for code in LESION_TYPES.values():
    (out_lesion_masks / code).mkdir(exist_ok=True)

# Also create merged/ subfolder (one combined mask per image)
(out_lesion_masks / 'merged').mkdir(exist_ok=True)

total_masks = {code: 0 for code in LESION_TYPES.values()}

for split_folder, tag in [('a. Training Set', 'train'),
                           ('b. Testing Set',  'test')]:
    for lesion_folder, short_code in LESION_TYPES.items():

        mask_dir = (Path(BASE_IDRID)
                    / '2. All Segmentation Groundtruths'
                    / split_folder
                    / lesion_folder)

        if not mask_dir.exists():
            print(f"⚠️  Not found: {mask_dir}")
            continue

        for mask_file in sorted(mask_dir.iterdir()):
            if mask_file.suffix.lower() in ('.png', '.jpg', '.jpeg', '.tif'):
                shutil.copy2(
                    mask_file,
                    out_lesion_masks / short_code / f'IDRID_{tag}_{mask_file.name}'
                )
                total_masks[short_code] += 1

print(f"✅ Lesion masks copied per type:")
for code, count in total_masks.items():
    print(f"    {code}  → {count} masks")
print(f"    ─────────────────────────────")
print(f"    TOTAL → {sum(total_masks.values())} masks")

✅ Lesion masks copied per type:
    MA  → 81 masks
    HE  → 80 masks
    EX  → 81 masks
    SE  → 40 masks
    OD  → 81 masks
    ─────────────────────────────
    TOTAL → 363 masks


In [9]:
import numpy as np
from PIL import Image

idrid_images = sorted([f for f in out_images.iterdir()
                        if f.name.startswith('IDRID')])
merged_count = 0

for img_file in idrid_images:
    # Extract the original IDRiD stem
    # filename format: IDRID_train_IDRiD_01.jpg → stem part is IDRiD_01
    parts = img_file.stem.split('_', 2)  # ['IDRID', 'train', 'IDRiD_01']
    if len(parts) < 3:
        continue
    tag        = parts[1]          # 'train' or 'test'
    idrid_stem = parts[2]          # 'IDRiD_01'

    combined = None

    for short_code in LESION_TYPES.values():
        mask_folder = out_lesion_masks / short_code
        for mask_file in mask_folder.iterdir():
            if idrid_stem.lower() in mask_file.stem.lower():
                arr = np.array(Image.open(mask_file).convert('L'))
                combined = arr if combined is None else np.maximum(combined, arr)
                break

    if combined is not None:
        save_path = out_lesion_masks / 'merged' / f'IDRID_{tag}_{idrid_stem}.png'
        Image.fromarray(combined).save(save_path)
        merged_count += 1

print(f"✅ Merged masks created: {merged_count}")

✅ Merged masks created: 81


In [12]:
import json

# ── Save README ─────────────────────────────────────────────────────────────
images       = list(out_images.iterdir())
vessel_masks = list(out_vessel_masks.iterdir())

retinomix_c = len([f for f in images if f.name.startswith('RETINOMIX')])
vessel_c    = len([f for f in images if f.name.startswith('VESSEL')])
idrid_c     = len([f for f in images if f.name.startswith('IDRID')])

lesion_counts = {}
for code in ['MA', 'HE', 'EX', 'SE', 'OD', 'merged']:
    folder = out_lesion_masks / code
    lesion_counts[code] = len(list(folder.iterdir())) if folder.exists() else 0

summary = {
    'project'      : 'AI Retinal Vessel Segmentation & DR Detection',
    'stage'        : 'Stage 1 — Vessel & Lesion Segmentation',
    'total_images' : len(images),
    'vessel_masks' : len(vessel_masks),
    'sources'      : {
        'RETINOMIX' : retinomix_c,
        'VESSEL'    : vessel_c,
        'IDRID'     : idrid_c
    },
    'lesion_masks' : lesion_counts
}

with open('/kaggle/working/dataset_stage1/README.json', 'w') as f:
    json.dump(summary, f, indent=2)

# ── Print audit ─────────────────────────────────────────────────────────────
vessel_total  = retinomix_c + vessel_c
od_count      = lesion_counts['OD']
merged_count  = lesion_counts['merged']

vessel_check  = '✅' if vessel_total == len(vessel_masks) else '❌  ' + str(vessel_total) + ' vs ' + str(len(vessel_masks))
od_check      = '✅' if idrid_c == od_count              else '❌  ' + str(idrid_c) + ' vs ' + str(od_count)
merged_check  = '✅' if idrid_c == merged_count           else '❌  ' + str(idrid_c) + ' vs ' + str(merged_count)

print("=" * 55)
print("STAGE 1 DATASET — COMPLETE FINAL AUDIT")
print("=" * 55)
print("\n  Images by source:")
print(f"    RETINOMIX   : {retinomix_c}")
print(f"    VESSEL      : {vessel_c}")
print(f"    IDRID       : {idrid_c}")
print(f"    {'─'*30}")
print(f"    TOTAL       : {len(images)}")
print(f"\n  Vessel masks  : {len(vessel_masks)}")
print(f"\n  Lesion masks by type:")
for code, count in lesion_counts.items():
    print(f"    {code:<8} : {count}")
print(f"\n  Pair checks:")
print(f"    RETINOMIX+VESSEL imgs == vessel masks : {vessel_check}")
print(f"    IDRID imgs == OD masks               : {od_check}")
print(f"    IDRID imgs == merged masks            : {merged_check}")
print(f"\n  README.json saved")
print(f"\n  Dataset is ready to save!")

STAGE 1 DATASET — COMPLETE FINAL AUDIT

  Images by source:
    RETINOMIX   : 233
    VESSEL      : 73
    IDRID       : 81
    ──────────────────────────────
    TOTAL       : 387

  Vessel masks  : 306

  Lesion masks by type:
    MA       : 81
    HE       : 80
    EX       : 81
    SE       : 40
    OD       : 81
    merged   : 81

  Pair checks:
    RETINOMIX+VESSEL imgs == vessel masks : ✅
    IDRID imgs == OD masks               : ✅
    IDRID imgs == merged masks            : ✅

  README.json saved

  Dataset is ready to save!
